# שלב 0 — notebook אימותים (קריאת התורה → כתוביות)

**איך מריצים (גם מדפדפן הטלפון):**
1. מלא את תא הפרמטרים שמתחת (ספר, פרק, ואם יש — נתיב ההקלטה ב-Google Drive).
2. תפריט **Runtime ← Run all** (בטלפון: ⋮ ← Runtime ← Run all).
3. גלול וקרא את הפלט — כל בדיקה מדפיסה ✅ או ❌ עם הסבר בעברית.

ה-notebook בודק את כל מה שתוכנית v5 (ראה `PLAN.md` בריפו) דורשת לאמת לפני ה-POC:
טעמים בטקסט של Sefaria, טבלת תדירויות הטעמים (שאלת הזרקא), קידוד קרי/כתיב,
פונקציית החיתוך על פרק אמיתי, וקריאות ההקלטה.

In [ ]:
#@title תא 1 — פרמטרים { display-mode: "form" }
BOOK = "Genesis"      #@param {type:"string"}  שם הספר באנגלית כמו ב-Sefaria (Genesis, Exodus, Leviticus, Numbers, Deuteronomy)
CHAPTER = 1           #@param {type:"integer"} מספר הפרק של ההקלטה
AUDIO_PATH = ""       #@param {type:"string"}  נתיב ההקלטה ב-Drive, למשל: /content/drive/MyDrive/kria.m4a  (ריק = דילוג על בדיקת האודיו)
BRANCH = "claude/project-action-plan-review-kyrf5x"  # ענף הקוד בריפו; אחרי מיזוג אפשר לשנות ל-main

print(f"ספר: {BOOK} | פרק: {CHAPTER}")
print("הקלטה:", AUDIO_PATH or "(לא הוגדרה — בדיקת האודיו תדולג)")

In [ ]:
#@title תא 2 — אימות: טקסט עם טעמים ב-Sefaria (גרסת MAM)
import re, requests, unicodedata

BASE = "https://www.sefaria.org"
CANT_RE = re.compile("[\u0591-\u05AF]")

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

# איתור גרסה עברית שמכילה "מסורה"/"Masorah" בשמה — לא מניחים שם מהזיכרון
versions = get_json(f"{BASE}/api/texts/versions/{BOOK}")
hebrew_versions = [v for v in versions if v.get("language") == "he"]
print("גרסאות עבריות זמינות בספר:")
mam_title = None
for v in hebrew_versions:
    title = v.get("versionTitle", "")
    print("  •", title)
    if "masorah" in title.lower() or "מסורה" in title:
        mam_title = title
if mam_title is None:
    raise SystemExit("❌ לא נמצאה גרסת MAM — בדוק את רשימת הגרסאות למעלה ועדכן ידנית")
print(f"\n✅ נמצאה גרסת MAM: «{mam_title}»")

# משיכת הפרק (API v3, עם פולבק ל-API הישן)
try:
    data = get_json(f"{BASE}/api/v3/texts/{BOOK} {CHAPTER}",
                    params={"version": f"hebrew|{mam_title}"})
    verses = data["versions"][0]["text"]
except Exception as e:
    print("אזהרה: v3 נכשל, עובר ל-API הישן.", e)
    data = get_json(f"{BASE}/api/texts/{BOOK}.{CHAPTER}", params={"vhe": mam_title})
    verses = data["he"]

print(f"\nנמשכו {len(verses)} פסוקים. פסוק ראשון כפי שהגיע (גולמי):\n{verses[0]}\n")

with_taamim = sum(1 for v in verses if CANT_RE.search(v))
print(f"פסוקים שמכילים טעמים: {with_taamim}/{len(verses)}")
print("✅ יש טעמים" if with_taamim == len(verses) else "❌ יש פסוקים בלי טעמים — עצור וחקור")

print("\nפירוק קודפוינטים של שתי המילים הראשונות (לזיהוי הטעמים בעין):")
for word in re.sub("<[^>]+>", "", verses[0]).split()[:2]:
    print(f"\n  {word}")
    for ch in word:
        print(f"    U+{ord(ch):04X}  {unicodedata.name(ch, '?')}")

In [ ]:
#@title תא 3 — טבלת תדירויות הטעמים על הספר כולו (מכריעה את שאלת הזרקא)
from collections import Counter

try:
    book_data = get_json(f"{BASE}/api/v3/texts/{BOOK}",
                         params={"version": f"hebrew|{mam_title}"})
    chapters = book_data["versions"][0]["text"]
except Exception as e:
    print("אזהרה: v3 נכשל, עובר ל-API הישן.", e)
    chapters = get_json(f"{BASE}/api/texts/{BOOK}", params={"vhe": mam_title})["he"]

counts = Counter()
double_in_word = Counter()
for chap in chapters:
    for verse in chap:
        for word in verse.split():
            marks = [c for c in word if CANT_RE.match(c)]
            counts.update(marks)
            for m, k in Counter(marks).items():
                if k > 1:
                    double_in_word[m] += 1

print(f"{'קודפוינט':<10}{'כמות':>8}{'  כפול-במילה':>12}   שם")
for ch, n in counts.most_common():
    print(f"U+{ord(ch):04X}    {n:>8}{double_in_word.get(ch, 0):>12}   {unicodedata.name(ch, '?')}")

z_0598, z_05ae = counts.get("\u0598", 0), counts.get("\u05AE", 0)
print(f"\nשאלת הזרקא: U+0598 (ZARQA) הופיע {z_0598} פעמים, U+05AE (ZINOR) הופיע {z_05ae} פעמים.")
print("⇒ הקודפוינט שבשימוש בפועל לזרקא הוא:",
      "U+05AE ✅ (כפי שחזתה v5)" if z_05ae > z_0598 else "U+0598 — לעדכן את PLAN.md!")
print("\nעמודת 'כפול-במילה' > 0 עבור פשטא/זרקא/תלישא ⇒ אישוש טעמי-העזר הכפולים של MAM (החיתוך כבר מטפל בזה)")

In [ ]:
#@title תא 4 — קידוד קרי/כתיב בפועל
# סריקה אמפירית: מאתרים פסוקים עם סוגריים/תגיות — כך מגלים איך MAM מסמן קרי/כתיב
import html as _html

candidates = []
for ci, chap in enumerate(chapters, 1):
    for vi, verse in enumerate(chap, 1):
        if re.search(r"[\[\]()<>]", _html.unescape(verse)):
            candidates.append((ci, vi, verse))

print(f"פסוקים עם סימון מיוחד (מועמדים לקרי/כתיב): {len(candidates)}")
for ci, vi, verse in candidates[:6]:
    print(f"\n{BOOK} {ci}:{vi}")
    print(verse[:300])
print("\nמסקנה נדרשת: איך מזהים את הקרי ואיך את הכתיב? עדכן את PLAN.md סעיף סיכון #4 בהתאם.")

In [ ]:
#@title תא 5 — הרצת פונקציית החיתוך על הפרק + בדיקת עיניים
import os, subprocess, sys

if not os.path.exists("Kriat-Hatora"):
    subprocess.run(["git", "clone", "-b", BRANCH,
                    "https://github.com/michaelnahmias1/Kriat-Hatora.git"], check=True)
sys.path.insert(0, "Kriat-Hatora/src")
from chunker import verse_to_segments, segments_for_pipeline

total = 0
for vi, verse in enumerate(verses, 1):
    segs = verse_to_segments(verse)
    total += len(segs)
    print(f"\n— פסוק {vi} —")
    for s in segs:
        print("    ", s)

print(f"\nסה\"כ {total} מקטעים ל-{len(verses)} פסוקים.")
print("מבחן הקבלה של שלב 1: קרא את המקטעים בעיניים — האם החיתוכים בגבולות תחביריים הגיוניים?")
print("\nדוגמה לפלט המלא של מקטע (תצוגה + שתי גרסאות יישור):")
print(segments_for_pipeline(verses[0])[0])

In [ ]:
#@title תא 6 — בדיקת ההקלטה (מדולג אם AUDIO_PATH ריק)
if not AUDIO_PATH:
    print("דילוג — לא הוגדר AUDIO_PATH בתא 1.")
else:
    import json as _json, subprocess
    if AUDIO_PATH.startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    probe = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json",
         "-show_format", "-show_streams", AUDIO_PATH],
        capture_output=True, text=True)
    info = _json.loads(probe.stdout)
    stream = next(s for s in info["streams"] if s["codec_type"] == "audio")
    dur = float(info["format"]["duration"])
    print(f"פורמט: {info['format']['format_name']} | קודק: {stream['codec_name']}")
    print(f"קצב דגימה: {stream['sample_rate']}Hz | ערוצים: {stream['channels']}")
    print(f"משך: {int(dur // 60)}:{dur % 60:04.1f} דקות")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", AUDIO_PATH,
                    "-ar", "16000", "-ac", "1", "recording_16k.wav"], check=True)
    print("\n✅ ההקלטה נקראת והומרה ל-recording_16k.wav (16kHz מונו) — מוכנה ל-POC של שלב 2")

## סיכום — מה הלאה

| בדיקה | תנאי מעבר |
|---|---|
| תא 2 | כל הפסוקים עם טעמים ✅ |
| תא 3 | טבלת התדירויות הגיונית; שאלת הזרקא הוכרעה |
| תא 4 | ברור איך מזוהה קרי/כתיב (או: אין בכלל בפרקים הרלוונטיים) |
| תא 5 | המקטעים נראים הגיוניים בעיניים |
| תא 6 | ההקלטה נקראת והומרה ל-16kHz מונו |

**הכל עבר?** שלב 0 + שלב 1 סגורים. הצעד הבא: notebook ה-POC של שלב 2 —
השוואה משולשת: MMS_FA+uroman (מנוקד) מול wav2vec2 עברי (לא מנוקד) מול baseline חלוקה-שווה.
ראה `PLAN.md` סעיף 5.